# Multinomial Naive Bayes: SMS spam classification

This notebook uses the UCI SMS Spam Collection from the `models.md` table to classify messages as ham or spam.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

url = 'https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip'
df = pd.read_csv(url, compression='zip', sep='\t', header=None, names=['label', 'text'])
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

# Fit the vocabulary on training messages only to avoid test-data leakage.
vectorizer = CountVectorizer()
X_train_counts = vectorizer.fit_transform(X_train)
X_test_counts = vectorizer.transform(X_test)

model = MultinomialNB(alpha=1.0)
model.fit(X_train_counts, y_train)

y_pred = model.predict(X_test_counts)
print(f'Accuracy:  {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred):.4f}')
print(f'F1 score:  {f1_score(y_test, y_pred):.4f}')
print('\nClassification report:\n', classification_report(y_test, y_pred, target_names=['ham', 'spam']))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=['ham', 'spam']).plot(cmap='Blues')
plt.title('Multinomial Naive Bayes: confusion matrix')
plt.show()

## Naive Bayes: model, smoothing, and evaluation

### Formula notation

- $c \in \{0,1\}$: class, where $0$ is ham and $1$ is spam.
- $\mathbf{x}$: bag-of-words vector for one message; $x_j$: count of vocabulary word $j$.
- $V$: vocabulary size; $N_{jc}$: total count of word $j$ in training messages of class $c$; $N_c$: total word count for class $c$.
- $\alpha$: Laplace smoothing value (`alpha=1.0`).

### Multinomial Naive Bayes model

Bayes' rule gives the class posterior:

$$P(c\mid\mathbf{x}) = \frac{P(\mathbf{x}\mid c)P(c)}{P(\mathbf{x})}$$

Naive Bayes assumes word counts are conditionally independent given the class. The smoothed probability of word $j$ in class $c$ is:

$$P(w_j\mid c) = \frac{N_{jc}+\alpha}{N_c+\alpha V}$$

The predicted class is the one with the largest log posterior:

$$\hat{c} = \operatorname*{argmax}_{c}\left[\log P(c) + \sum_{j=1}^{V}x_j\log P(w_j\mid c)\right]$$

Log probabilities turn many probability multiplications into stable additions. Laplace smoothing is used so an unseen word does not make an entire class probability zero.

### Test-set evaluation

$$\mathrm{Accuracy} = \frac{TP+TN}{TP+TN+FP+FN}$$

$$\mathrm{Precision} = \frac{TP}{TP+FP}, \qquad \mathrm{Recall} = \frac{TP}{TP+FN}$$

$$F_1 = 2\cdot\frac{\mathrm{Precision}\cdot\mathrm{Recall}}{\mathrm{Precision}+\mathrm{Recall}}$$

**Maximize** Accuracy, Precision, Recall, and $F_1$: each ranges from $0$ (worst) to $1$ (best). For spam filtering, spam recall measures missed spam and spam precision measures how often messages flagged as spam truly are spam.